In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import psycopg2
from datetime import datetime
import talib as ta

In [21]:
from config import db_conn
conn = db_conn()

symbol = 'FEDERALBNK-I'
from_date = '2025-10-01'
to_date = '2025-10-31'

query = f"""
    SELECT date AT TIME ZONE 'Asia/Kolkata' AS local_time, *
    FROM idata_15min
    WHERE symbol = %s AND date(date) >= %s AND date(date) <= %s
    ORDER BY date ASC;
"""
params = (symbol, from_date, to_date)

cursor = conn.cursor()
cursor.execute(query, params)
rows = cursor.fetchall()
columns = [desc[0] for desc in cursor.description]

df = pd.DataFrame(rows, columns=columns)
df['date'] = df["local_time"]
df.sort_values(by="date", inplace=True)
# df["date"] = pd.to_datetime(df["date"])
df.drop(columns=["local_time", "id"], inplace=True)
numeric_cols = ["open", "high", "low", "close", "volume"]
df[numeric_cols] = df[numeric_cols].astype(float)



In [28]:
df["vwap"] = (df["close"] * df["volume"]).cumsum() / df["volume"].cumsum()
df["ema13"] = ta.EMA(df["close"], timeperiod=13)
df["ema34"] = ta.EMA(df["close"], timeperiod=34)
df["rsi14"] = ta.RSI(df["close"], timeperiod=14)

# If your dataset has multiple days, reset per day:
df["vwap"] = df.groupby(df["date"].dt.date).apply(
    lambda x: (x["close"] * x["volume"]).cumsum() / x["volume"].cumsum()
).reset_index(level=0, drop=True)

df.dropna(inplace=True)

df["above_vwap"] = df["close"] > df["vwap"]
df["momentum"] = (df["ema13"] > df["ema34"]) & (df["rsi14"] > 60)
df["vwap_breakout"] = (
    df["above_vwap"] &
    (df["close"] > df["high"].shift(1)) &
    df["momentum"]
)



In [30]:
df[df["vwap_breakout"]][["date", "close", "vwap", "vwap_breakout"]]

,date,close,vwap,vwap_breakout
114,2025-10-08 12:45:00,201.64,201.616519,True
115,2025-10-08 13:00:00,202.19,201.638996,True
116,2025-10-08 13:15:00,203.01,201.748877,True
117,2025-10-08 13:30:00,203.67,202.019129,True
127,2025-10-09 09:45:00,205.13,204.686873,True
128,2025-10-09 10:00:00,206.59,205.214351,True
129,2025-10-09 10:15:00,207.89,205.770558,True
134,2025-10-09 11:30:00,207.24,206.237044,True
135,2025-10-09 11:45:00,207.44,206.277172,True
141,2025-10-09 13:15:00,207.87,206.406993,True
